In [26]:
import pandas as pd
import numpy as np
from pathlib import Path 

## Merge firm data

In [27]:
dir = "/Users/yutung/MQF/Machine learning/Project/ml_project/data/processed/"

monthly_stock = pd.read_parquet(dir + "monthly_stock.parquet")
print(monthly_stock.head())

fund_monthly = pd.read_parquet(dir + "fundamentals_monthly.parquet")
print(fund_monthly.head())


# align keys (LPERMNO -> PERMNO, datadate -> date)
fund_monthly = fund_monthly.rename(
    columns={"LPERMNO": "PERMNO", "datadate": "date"}
)

# make sure date columns are datetime
fund_monthly["date"] = pd.to_datetime(fund_monthly["date"])
monthly_stock["date"] = pd.to_datetime(monthly_stock["date"])

# merge on PERMNO + date
merged_1 = monthly_stock.merge(
    fund_monthly,
    on=["PERMNO", "date"],
    how="left"
)

print(merged_1.head())
print(merged_1.shape)

   PERMNO       date  stock_ret  stock_vol        dvol  turnover  \
0   10001 2000-01-31  -0.044119   0.025332  335742.750  0.823122   
1   10001 2000-02-29   0.015383   0.012548  181390.625  0.452571   
2   10001 2000-03-31  -0.015289   0.026495  581272.875  1.283407   
3   10001 2000-04-30   0.011720   0.022529  211757.625  0.562628   
4   10001 2000-05-31  -0.023166   0.016333  178489.375  0.407984   

   stock_mktcap    bidask  numtrades  stock_price_mean  
0      19906.25  0.023530       97.0          8.293750  
1      20212.50  0.013686       60.0          8.184375  
2      19712.00  0.018115       98.0          7.970109  
3      19943.00  0.018016       57.0          8.052631  
4      19481.00  0.020593       49.0          8.053977  
   LPERMNO   datadate  gsector      cusip   log_atq  lev_total  equity_ratio  \
0    10001 1990-03-31     55.0  367204104  3.007908   0.642233      0.357767   
1    10001 1990-04-30     55.0  367204104  3.007908   0.642233      0.357767   
2    1000

## Merge macro data

In [28]:
macro = pd.read_excel(dir + "macro_data.xlsx")
print(macro.head())
# convert date to datetime
macro["date"] = pd.to_datetime(macro["date"].astype(str))

# create macro features
macro["sp500_ret"] = macro["sp500"].pct_change().fillna(0)
macro["ir3m_chg"] = macro["ir3m"].diff().fillna(0)
macro["ir10y_chg"] = macro["ir10y"].diff().fillna(0)
macro["vix_chg"] = macro["vix"].pct_change().fillna(0)
macro["gdp_gr"] = macro["gdp"].pct_change().fillna(0)
macro["cpi_infl"] = macro["cpi"].pct_change().fillna(0)

print(macro.head())


         sp500  ir3m  ir10y        vix        gdp    cpi      date
0   980.280029  5.04  5.512  21.469999  12703.742  162.0  19980131
1  1049.339966  5.18  5.616  18.549999  12703.742  162.0  19980228
2  1101.750000  4.99  5.662  24.219999  12703.742  162.0  19980331
3  1111.750000  4.85  5.667  21.180000  12821.339  162.2  19980430
4  1090.819946  4.89  5.546  21.320000  12821.339  162.6  19980531
         sp500  ir3m  ir10y        vix        gdp    cpi       date  \
0   980.280029  5.04  5.512  21.469999  12703.742  162.0 1998-01-31   
1  1049.339966  5.18  5.616  18.549999  12703.742  162.0 1998-02-28   
2  1101.750000  4.99  5.662  24.219999  12703.742  162.0 1998-03-31   
3  1111.750000  4.85  5.667  21.180000  12821.339  162.2 1998-04-30   
4  1090.819946  4.89  5.546  21.320000  12821.339  162.6 1998-05-31   

   sp500_ret  ir3m_chg  ir10y_chg   vix_chg    gdp_gr  cpi_infl  
0   0.000000      0.00      0.000  0.000000  0.000000  0.000000  
1   0.070449      0.14      0.104 -0.13

/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_35438/3291528241.py:11: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  macro["gdp_gr"] = macro["gdp"].pct_change().fillna(0)
/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_35438/3291528241.py:12: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  macro["cpi_infl"] = macro["cpi"].pct_change().fillna(0)


In [29]:
# sort for safety
macro = macro.sort_values("date")

# ensure merged (stock+fund) also uses datetime for date
merged_1["date"] = pd.to_datetime(merged_1["date"])

# merge macro into merged dataset
merged_2 = merged_1.merge(
    macro,
    on="date",
    how="left"
)

unused_macro_cols = ["sp500", "ir3m", "ir10y", "vix", "gdp", "cpi"]
# drop unused macro columns
merged_2 = merged_2.drop(columns=unused_macro_cols, errors="ignore")

print(merged_2.head())
print("shape:", merged_2.shape)

   PERMNO       date  stock_ret  stock_vol        dvol  turnover  \
0   10001 2000-01-31  -0.044119   0.025332  335742.750  0.823122   
1   10001 2000-02-29   0.015383   0.012548  181390.625  0.452571   
2   10001 2000-03-31  -0.015289   0.026495  581272.875  1.283407   
3   10001 2000-04-30   0.011720   0.022529  211757.625  0.562628   
4   10001 2000-05-31  -0.023166   0.016333  178489.375  0.407984   

   stock_mktcap    bidask  numtrades  stock_price_mean  ...  market_to_book  \
0      19906.25  0.023530       97.0          8.293750  ...        1.592727   
1      20212.50  0.013686       60.0          8.184375  ...        1.592727   
2      19712.00  0.018115       98.0          7.970109  ...        1.386411   
3      19943.00  0.018016       57.0          8.052631  ...        1.386411   
4      19481.00  0.020593       49.0          8.053977  ...        1.386411   

  atq_growth  revtq_growth  niq_growth  sp500_ret  ir3m_chg  ir10y_chg  \
0   0.099523      0.521947   -1.750751  -0

In [30]:
industry = pd.read_excel(dir + "industry_data.xlsx")

# convert date column
industry["date"] = pd.to_datetime(industry["date"].astype(str))

# rename ETF columns to avoid confusion
industry = industry.rename(columns={
    "sector": "sector_etf",
    "price": "etf_price",
    "return": "etf_return"
})

# sector mapping
sector_map = {
    10: "XLE",
    15: "XLB",
    20: "XLI",
    25: "XLY",
    30: "XLP",
    35: "XLV",
    40: "XLF",
    45: "XLK",
    50: "XLC",
    55: "XLU",
    60: "XLRE",
}

# add ETF ticker to merged via gsector
merged_2["sector_etf"] = merged_2["gsector"].map(sector_map)

# merge ETF monthly data on (sector_etf, date)
industry = industry.rename(columns={"sector": "sector_etf"})

merged_3 = merged_2.merge(
    industry,
    on=["sector_etf", "date"],
    how="left"
)

print(merged_3.head())
print("shape:", merged_3.shape)

   PERMNO       date  stock_ret  stock_vol        dvol  turnover  \
0   10001 2000-01-31  -0.044119   0.025332  335742.750  0.823122   
1   10001 2000-02-29   0.015383   0.012548  181390.625  0.452571   
2   10001 2000-03-31  -0.015289   0.026495  581272.875  1.283407   
3   10001 2000-04-30   0.011720   0.022529  211757.625  0.562628   
4   10001 2000-05-31  -0.023166   0.016333  178489.375  0.407984   

   stock_mktcap    bidask  numtrades  stock_price_mean  ...  niq_growth  \
0      19906.25  0.023530       97.0          8.293750  ...   -1.750751   
1      20212.50  0.013686       60.0          8.184375  ...   -1.750751   
2      19712.00  0.018115       98.0          7.970109  ...    1.882000   
3      19943.00  0.018016       57.0          8.052631  ...    1.882000   
4      19481.00  0.020593       49.0          8.053977  ...    1.882000   

  sp500_ret  ir3m_chg  ir10y_chg   vix_chg    gdp_gr  cpi_infl  sector_etf  \
0 -0.050904      0.36      0.232  0.012581  0.003628  0.002962

## Merge bond data

In [31]:
bond_data = pd.read_parquet(dir + "bond_data_processed.parquet")
print(bond_data.head())

        date      cusip company_symbol   tmt   coupon  t_spread    yield  \
0 2002-07-31  000325AA8           AAFM  0.55  0.08875       NaN      NaN   
1 2002-08-31  000325AA8           AAFM  0.47  0.08875    0.0042  0.07731   
2 2002-09-30  000325AA8           AAFM  0.38  0.08875    0.0078  0.07933   
3 2002-10-31  000325AA8           AAFM  0.30  0.08875       NaN  0.07708   
4 2002-11-30  000325AA8           AAFM  0.21  0.08875    0.0025  0.04793   

    ret_eom  rating_A  rating_AA  ...  rating_BB  rating_BBB  rating_C  \
0       NaN       NaN        NaN  ...        NaN         NaN       NaN   
1  0.005162       0.0        0.0  ...        0.0         0.0       0.0   
2  0.007239       0.0        0.0  ...        0.0         0.0       0.0   
3  0.014820       0.0        0.0  ...        0.0         0.0       0.0   
4 -0.000199       0.0        0.0  ...        0.0         0.0       0.0   

   rating_CC  rating_CCC  rating_D  upgrade  downgrade    gs3m  term_spread  
0        NaN        

In [32]:
def normalize_cusip6(s):
    s = s.astype(str).str.strip().str.upper()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.pad(6, fillchar='0')
    return s.str[:6]

bond_data['issuer6'] = normalize_cusip6(bond_data['cusip'])

# find stock CUSIP root
if 'cusip' in merged_3.columns:
    merged_3['issuer6'] = normalize_cusip6(merged_3['cusip'])
elif 'cusip_firm' in merged_3.columns:
    merged_3['issuer6'] = normalize_cusip6(merged_3['cusip_firm'])
else:
    print("merged_3 has no cusip column!")

bond_data['month'] = bond_data['date'].dt.to_period('M')
merged_3['month']  = merged_3['date'].dt.to_period('M')



In [33]:
# find common issuer
bond_issuers = set(bond_data["issuer6"].dropna().unique())
merged_issuers = set(merged_3["issuer6"].dropna().unique())
common_issuers = bond_issuers & merged_issuers

print("bond issuers :", len(bond_issuers))
print("merged issuers:", len(merged_issuers))
print("valid issuers:", len(common_issuers))


bond issuers : 5699
merged issuers: 16280
valid issuers: 1486


In [34]:
print("atq" in merged_3.columns)
print(sorted(merged_3.columns))

False
['PERMNO', 'atq_growth', 'bidask', 'cpi_infl', 'cusip', 'date', 'dvol', 'equity_ratio', 'etf_price', 'etf_return', 'gdp_gr', 'gsector', 'int_coverage', 'ir10y_chg', 'ir3m_chg', 'issuer6', 'lev_total', 'log_atq', 'market_to_book', 'mkt_cap', 'month', 'niq_growth', 'numtrades', 'profit_margin', 'revtq_growth', 'roa', 'sector_etf', 'sp500_ret', 'stock_mktcap', 'stock_price_mean', 'stock_ret', 'stock_vol', 'turnover', 'vix_chg']


In [36]:
# select bond_data
bond_filtered = bond_data[bond_data["issuer6"].isin(common_issuers)].copy()

print("before:", bond_data.shape)
print("after :", bond_filtered.shape)

# shift features in merged_3 since bond features are lagged by 1 month
cols_to_shift = [

    # stock
    "stock_ret", "stock_vol", "dvol", "turnover",
    "stock_mktcap", "bidask", "numtrades", "stock_price_mean",

    # derived
    "log_atq","lev_total","equity_ratio","roa",
    "profit_margin","int_coverage","mkt_cap",
    "market_to_book",
    "atq_growth","revtq_growth","niq_growth",

    # macro
    "sp500_ret", "ir3m_chg", "ir10y_chg",
    "vix_chg", "gdp_gr", "cpi_infl",

    # industry
    "etf_price","etf_return",
]

merged_3_lag = (
    merged_3.sort_values(["issuer6","date"])
    .groupby("issuer6", group_keys=False)
    .apply(lambda g: g.assign(**{c: g[c].shift(1) for c in cols_to_shift}))
)


print("merged_3：")
print(merged_3.loc[merged_3["issuer6"] == list(common_issuers)[0],
                   ["issuer6", "date"] + cols_to_shift[:3]].head(5))

print("\n merged_3_lag：")
print(merged_3_lag.loc[merged_3_lag["issuer6"] == list(common_issuers)[0],
                       ["issuer6", "date"] + cols_to_shift[:3]].head(5))


before: (3567979, 24)
after : (871712, 24)
merged_3：
       issuer6       date  stock_ret  stock_vol         dvol
108549  03769M 2011-03-31  -0.010989   0.007813  423639168.0
108550  03769M 2011-04-30   0.004445   0.006904  294559264.0
108551  03769M 2011-05-31   0.009020   0.018064  211916672.0
108552  03769M 2011-06-30  -0.045504   0.019718   89744624.0
108553  03769M 2011-07-31   0.006394   0.017465   55737736.0

 merged_3_lag：
       issuer6       date  stock_ret  stock_vol         dvol
108549  03769M 2011-03-31        NaN        NaN          NaN
108550  03769M 2011-04-30  -0.010989   0.007813  423639168.0
108551  03769M 2011-05-31   0.004445   0.006904  294559264.0
108552  03769M 2011-06-30   0.009020   0.018064  211916672.0
108553  03769M 2011-07-31  -0.045504   0.019718   89744624.0


/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_35438/1233077242.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.assign(**{c: g[c].shift(1) for c in cols_to_shift}))


In [38]:
merged_all = bond_filtered.merge(
    merged_3_lag,
    on=["issuer6", "date"],
    how="left"
)

print(merged_all.head())

summary = merged_all.describe().loc[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
].T

summary

        date    cusip_x company_symbol   tmt  coupon  t_spread    yield  \
0 2002-07-31  000361AB1            AIR  1.23  0.0725       NaN      NaN   
1 2002-08-31  000361AB1            AIR  1.14  0.0725       NaN  0.04827   
2 2002-09-30  000361AB1            AIR  1.06  0.0725       NaN  0.04386   
3 2002-10-31  000361AB1            AIR  0.97  0.0725       NaN  0.04122   
4 2002-11-30  000361AB1            AIR  0.89  0.0725       NaN  0.03873   

    ret_eom  rating_A  rating_AA  ...  sp500_ret  ir3m_chg  ir10y_chg  \
0       NaN       NaN        NaN  ...  -0.072455    -0.046     -0.219   
1  0.008709       0.0        0.0  ...  -0.079004     0.006     -0.359   
2  0.006141       0.0        0.0  ...   0.004881    -0.020     -0.328   
3  0.005690       0.0        0.0  ...  -0.110024    -0.118     -0.530   
4  0.001961       0.0        0.0  ...   0.086449    -0.110      0.304   

    vix_chg    gdp_gr  cpi_infl  sector_etf  etf_price  etf_return  month_y  
0  0.271271  0.000000  0.000557 

,mean,std,min,25%,50%,75%,max
date,2015-06-06 02:15:24.712105984,NaN,2002-07-31 00:00:00,2010-09-30 00:00:00,2016-05-31 00:00:00,2020-07-31 00:00:00,2024-08-31 00:00:00
tmt,10.203738,10.775051,0.0,3.26,6.48,14.33,101.46
coupon,0.051542,0.02072,0.0,0.0365,0.05125,0.0665,0.21
t_spread,0.005062,0.009505,0.0,0.0016,0.0031,0.0059,1.967
yield,0.043845,0.042995,-1.0,0.02724,0.04217,0.05561,0.9998
ret_eom,0.005801,0.394508,-0.991,-0.004602,0.00342,0.01269,111.81
rating_A,0.281419,0.449691,0.0,0.0,0.0,1.0,1.0
rating_AA,0.06635,0.248893,0.0,0.0,0.0,0.0,1.0
rating_AAA,0.012533,0.111247,0.0,0.0,0.0,0.0,1.0
rating_B,0.038539,0.192494,0.0,0.0,0.0,0.0,1.0


In [ ]:
dir = "/Users/yutung/MQF/Machine learning/Project/ml_project/data/"
merged_all.to_parquet(dir + "merged_all.parquet", index=False)